<a href="https://colab.research.google.com/github/Vidushi0907/Machine-Learning-/blob/main/Case_Study_4_Ecommerce_Customer_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 4: E-Commerce Customer Segmentation
**MLE Exam | RFM Analysis using K-Means, DBSCAN, PCA, and t-SNE**

## Aim
To segment customers of an online retailer using Recency, Frequency, and Monetary (RFM) features, compare K-Means and DBSCAN clustering, reduce dimensions using PCA, visualize clusters using t-SNE, and suggest marketing personas.

**Kaggle dataset:** [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail)

### Dataset note
This notebook uses the Online Retail transaction dataset. It expects a file named `OnlineRetail.csv` (the Kaggle download may have a slightly different filename). Upload the CSV when prompted.

### Workflow
1. Load and clean transaction data.
2. Create RFM features for each customer.
3. Scale RFM features.
4. Apply K-Means and DBSCAN.
5. Use PCA for dimensionality reduction and t-SNE for visualization.
6. Interpret clusters and derive marketing personas.

**Note:** Clustering is unsupervised learning, so there are no correct target labels. The quality and usefulness of clusters depend on data cleaning, feature choices, and parameter settings.

## 1. Install and import libraries

In [ ]:
!pip -q install scikit-learn seaborn openpyxl

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

print("Libraries imported successfully.")

## 2. Upload and load the Kaggle dataset

In [ ]:
# Upload the CSV downloaded from Kaggle
uploaded = files.upload()
file_name = next(iter(uploaded))

# Online Retail datasets are commonly CSV files; try common encodings
try:
    retail = pd.read_csv(file_name, encoding="ISO-8859-1")
except UnicodeDecodeError:
    retail = pd.read_csv(file_name, encoding="utf-8", errors="replace")

print("Dataset shape:", retail.shape)
print("Columns:", retail.columns.tolist())
retail.head()

## 3. Clean data and create RFM features

- **Recency:** Number of days since the customer's last purchase (lower is more recent).
- **Frequency:** Number of distinct invoices/orders made by the customer.
- **Monetary:** Total spending by the customer.

We remove cancelled invoices, missing customer IDs, and non-positive quantities/prices. We use the latest transaction date in the dataset plus one day as the reference date, so this is a reproducible snapshot rather than today's real-world recency.

In [ ]:
# Standardize column names in case capitalization differs
retail.columns = retail.columns.str.strip()

required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID"
]
missing = [col for col in required_columns if col not in retail.columns]
if missing:
    raise ValueError(
        f"Missing expected columns: {missing}. "
        "Please use the Kaggle Online Retail dataset with InvoiceNo, InvoiceDate, "
        "Quantity, UnitPrice, and CustomerID columns."
    )

# Remove rows without customer IDs
retail = retail.dropna(subset=["CustomerID"]).copy()

# Remove cancelled invoices (invoice numbers beginning with C)
retail["InvoiceNo"] = retail["InvoiceNo"].astype(str)
retail = retail[~retail["InvoiceNo"].str.startswith("C")]

# Remove returns/invalid transactions
retail = retail[(retail["Quantity"] > 0) & (retail["UnitPrice"] > 0)]

# Convert dates and calculate transaction value
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail = retail.dropna(subset=["InvoiceDate"])
retail["CustomerID"] = retail["CustomerID"].astype(str)
retail["TotalAmount"] = retail["Quantity"] * retail["UnitPrice"]

# Reference date is one day after the last transaction in this dataset
reference_date = retail["InvoiceDate"].max() + pd.Timedelta(days=1)

# Create RFM table
rfm = retail.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda dates: (reference_date - dates.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalAmount", "sum")
).reset_index()

# Remove invalid values if any
rfm = rfm[(rfm["Frequency"] > 0) & (rfm["Monetary"] > 0)].copy()

print("Number of customers:", len(rfm))
rfm.head()

## 4. Explore and prepare RFM data
RFM features are often skewed: a small number of customers may order very often or spend much more than others. We use `log1p` to reduce skew and then standardize features so they are on comparable scales.

In [ ]:
# Summary statistics
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

# Visualize distributions
for column in ["Recency", "Frequency", "Monetary"]:
    plt.figure(figsize=(6, 3))
    plt.hist(rfm[column], bins=40)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Number of customers")
    plt.tight_layout()
    plt.show()

# Log transform to reduce the effect of extreme values
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()
rfm_log = np.log1p(rfm_features)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log)

print("Scaled RFM data shape:", X_scaled.shape)

## 5. K-Means clustering
K-Means divides customers into a chosen number of clusters. We'll try several values of K and use the elbow curve as a simple guide. The elbow is a heuristic, not a guaranteed best choice.

In [ ]:
inertias = []
k_values = range(2, 8)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

In [ ]:
# Choose K=4 for this practical example.
# If your elbow curve suggests another value, you can change it.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)

print("K-Means cluster sizes:")
print(rfm["KMeans_Cluster"].value_counts().sort_index())

# Silhouette score: higher is generally better, if there is more than one cluster
print("K-Means silhouette score:",
      round(silhouette_score(X_scaled, rfm["KMeans_Cluster"]), 3))

## 6. DBSCAN clustering
DBSCAN groups dense regions and can identify outliers as noise (label `-1`). Unlike K-Means, we do not specify the number of clusters beforehand. Results depend strongly on `eps` and `min_samples`.

In [ ]:
# These are starter settings; DBSCAN may need tuning for your dataset.
dbscan = DBSCAN(eps=0.8, min_samples=10)
rfm["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

print("DBSCAN cluster sizes (label -1 means noise):")
print(rfm["DBSCAN_Cluster"].value_counts().sort_index())

# Silhouette score is only defined if there are 2 or more clusters,
# and noise points are excluded for this simple comparison.
db_labels = rfm["DBSCAN_Cluster"].to_numpy()
non_noise = db_labels != -1
unique_clusters = set(db_labels[non_noise])

if len(unique_clusters) >= 2 and non_noise.sum() > len(unique_clusters):
    score = silhouette_score(X_scaled[non_noise], db_labels[non_noise])
    print("DBSCAN silhouette score (excluding noise):", round(score, 3))
else:
    print("DBSCAN silhouette score not calculated: try tuning eps/min_samples.")

## 7. PCA: reduce RFM to two dimensions
PCA creates new features that capture as much variance as possible in fewer dimensions. We use two principal components to make a simple 2D plot. PCA is a linear dimensionality reduction method.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))
print("Total variance explained by 2 components:",
      round(pca.explained_variance_ratio_.sum(), 3))

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=rfm["KMeans_Cluster"], s=12
)
plt.title("K-Means Customer Segments (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="K-Means Cluster")
plt.tight_layout()
plt.show()

## 8. t-SNE visualization
t-SNE is a nonlinear technique that tries to preserve local neighborhoods in a low-dimensional view. It is useful for visualization, but distances between far-apart groups and the apparent size of clusters should not be over-interpreted. It is not used to train the clustering models here.

In [ ]:
# t-SNE can take time on large datasets.
# Use a fixed random_state for a reproducible visualization.
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42
)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=rfm["KMeans_Cluster"], s=12
)
plt.title("K-Means Customer Segments (t-SNE)")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.colorbar(scatter, label="K-Means Cluster")
plt.tight_layout()
plt.show()

## 9. Interpret clusters and create marketing personas
Cluster IDs are arbitrary: cluster 0 is not automatically better than cluster 1. We interpret them by looking at average Recency, Frequency, and Monetary values in the original units.

In [ ]:
# Average RFM values for each K-Means cluster
kmeans_summary = rfm.groupby("KMeans_Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency_Days=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).round(2)

display(kmeans_summary)

# Build simple personas from relative cluster behavior.
# Thresholds use the overall median so the labels adapt to this dataset.
median_recency = rfm["Recency"].median()
median_frequency = rfm["Frequency"].median()
median_monetary = rfm["Monetary"].median()

def create_persona(row):
    recent = row["Avg_Recency_Days"] <= median_recency
    frequent = row["Avg_Frequency"] >= median_frequency
    high_spend = row["Avg_Monetary"] >= median_monetary

    if recent and frequent and high_spend:
        return "VIP Loyalists"
    elif recent and not frequent:
        return "Recent / New Customers"
    elif not recent and frequent:
        return "At-Risk Repeat Customers"
    elif not recent and not frequent:
        return "Dormant / Low-Engagement Customers"
    else:
        return "Regular Customers"

kmeans_summary["Marketing_Persona"] = kmeans_summary.apply(create_persona, axis=1)
display(kmeans_summary)

### Suggested marketing actions
Use these as initial hypotheses and validate them against campaign outcomes.

- **VIP Loyalists:** loyalty rewards, early access, referral incentives.
- **Recent / New Customers:** onboarding emails, product recommendations, second-purchase offers.
- **At-Risk Repeat Customers:** personalized win-back offers and reminders.
- **Dormant / Low-Engagement Customers:** low-cost reactivation campaigns; avoid excessive discounting.
- **Regular Customers:** cross-selling, bundles, and relevant recommendations.

A cluster can receive a different persona depending on its measured RFM profile. Always inspect the summary table before acting on the labels.

## 10. Compare K-Means and DBSCAN

In [ ]:
print("K-Means clusters (excluding noise concept):",
      rfm["KMeans_Cluster"].nunique())
print("DBSCAN clusters (excluding noise):",
      len(set(rfm["DBSCAN_Cluster"]) - {-1}))
print("DBSCAN noise customers:",
      int((rfm["DBSCAN_Cluster"] == -1).sum()))

comparison = pd.DataFrame({
    "Method": ["K-Means", "DBSCAN"],
    "Number of clusters": [
        rfm["KMeans_Cluster"].nunique(),
        len(set(rfm["DBSCAN_Cluster"]) - {-1})
    ],
    "Noise points": [
        0,
        int((rfm["DBSCAN_Cluster"] == -1).sum())
    ]
})
display(comparison)

## 11. Learning efficiency and real-world deployment constraints

### K-Means
- Efficient and easy to implement on many datasets.
- Requires the number of clusters (`K`) in advance.
- Sensitive to scaling, outliers, and initial centroids.
- Works best when clusters are reasonably compact and separated.

### DBSCAN
- Does not require the number of clusters in advance.
- Can identify outliers/noise and clusters with irregular shapes.
- Sensitive to `eps` and `min_samples`.
- May struggle when cluster densities vary or feature dimensions are high.

### PCA and t-SNE
- **PCA** is relatively fast and gives a compact linear representation; explained variance helps summarize how much information the two components retain.
- **t-SNE** is mainly for visualizing local neighborhood structure. It can be slower, sensitive to settings, and should not be treated as proof that clusters are objectively distinct.

### Deployment constraints
- RFM values change as new purchases arrive, so segmentation needs periodic updates.
- High-volume retailers may need scalable pipelines and scheduled recomputation.
- Marketing personas must be interpretable and validated with campaign experiments.
- Customer data must be handled securely and in line with privacy requirements.
- The clustering result is only as useful as the data quality and the business action it supports.

## 12. Conclusion
This case study used the Kaggle Online Retail transaction dataset to create Recency, Frequency, and Monetary features for customers. K-Means and DBSCAN were applied to segment customers, PCA reduced the RFM data to two dimensions, and t-SNE provided another visualization of customer groupings. Cluster summaries were used to suggest marketing personas and actions.

K-Means is a simple, efficient choice when the desired number of groups is known, while DBSCAN can identify dense groups and noise without specifying the number of clusters. The best method should be chosen using cluster quality, stability, interpretability, scalability, and usefulness for marketing—not visual appearance alone.

## 13. Viva questions
1. **What is customer segmentation?** Grouping customers with similar behavior or characteristics.
2. **What does RFM stand for?** Recency, Frequency, and Monetary value.
3. **What is Recency?** How many days have passed since the customer's most recent purchase.
4. **What is Frequency?** How often the customer purchases, measured here as distinct invoices.
5. **What is Monetary value?** Total transaction spending for the customer.
6. **Is clustering supervised or unsupervised?** Unsupervised learning, because there are no target labels.
7. **How does K-Means work?** It assigns points to the nearest centroid and repeatedly updates centroids to reduce within-cluster squared distances.
8. **What is a limitation of K-Means?** The number of clusters must be selected, and it can be affected by outliers.
9. **What is DBSCAN?** A density-based clustering method that can label outliers as noise.
10. **What does DBSCAN label `-1` mean?** The point was classified as noise/outlier under the chosen parameters.
11. **What is PCA?** A linear method that projects data onto components capturing high variance.
12. **What is t-SNE?** A nonlinear technique commonly used to visualize high-dimensional data in two dimensions.
13. **Why scale RFM features?** To stop features with large numeric ranges from dominating distance calculations.
14. **Why use log transformation?** To reduce skew and lessen the influence of extreme spending or frequency values.
15. **How can a retailer use the clusters?** For loyalty campaigns, reactivation, cross-selling, and personalized offers.

**Exam tip:** Run this notebook before your practical. If DBSCAN returns one cluster or mostly noise, try adjusting `eps` (for example, 0.6, 0.8, 1.0, or 1.2) and inspect the cluster counts. Don't assume a parameter setting will work equally well for every dataset.